# ASD variants: MutPred2 coverage from Tim's BrainSpan-isoform file

**Question 1.** Of the variants on the `ASD` sheet of `extractedVariants.xlsx`, how many can we score
using the new file Tim sent (`MutPred2_scores_ASD_brainspan.tsv` — MutPred2 scores computed on the
BrainSpan isoforms)?

**Question 2.** How many of those ASD variants have a MutPred2 score in Tulika's file
(`mutpred_score_variant_transcript_tulika.tsv`)?

Tulika's count is **reference only** — it tells us what her set covered, and so which ASD variants we
may still be missing. It is not counted as our coverage.

A variant counts as *scored* if at least one of its variant x transcript rows has a non-null score.
Self-contained: reads only files under `data/`.

## 1. Setup — file paths

In [1]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, ExtractedVariants_file, BrainSpan_expression_file_cortical

DATA        = Path(str(DATA_DIR))
TIM_ASD     = DATA / "MutPred2_scores" / "MutPred2_scores_ASD_brainspan.tsv"
TULIKA_FILE = DATA / "MutPred2_scores" / "mutpred_score_variant_transcript_tulika.tsv"

for f in (ExtractedVariants_file, TIM_ASD, TULIKA_FILE, BrainSpan_expression_file_cortical):
    print(f"{'OK ' if Path(f).exists() else 'MISSING'}  {f}")

OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/extractedVariants.xlsx
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_scores_ASD_brainspan.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/mutpred_score_variant_transcript_tulika.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/raw/brainspan/BrainSpan_FuSatt_meanTPM_byRegion_byPeriod_Cortical_Chau11.tsv


## 2. Load the ASD sheet

One row per variant per status. The `Variant` column (`chr_pos_REF/ALT`, hg38) is the key used to
match against both score files. A handful of variants appear twice — once as `ASD` (case) and once as
`Control` — so we count distinct variants.

In [2]:
asd = pd.read_excel(ExtractedVariants_file, sheet_name="ASD")
ASD_VARIANTS = set(asd["Variant"])

# sanity check: the Variant key agrees with the coordinate columns
key = asd["chr"].astype(str) + "_" + asd["hg38_start"].astype(str) + "_" + asd["REF"] + "/" + asd["ALT"]
assert (key == asd["Variant"]).all()

print(f"ASD sheet rows        : {len(asd)}")
print(f"distinct variants     : {len(ASD_VARIANTS)}")
print(f"rows by Status        : {asd['Status'].value_counts().to_dict()}")
both = asd.groupby("Variant")["Status"].nunique().loc[lambda s: s > 1].index.tolist()
print(f"listed as both ASD and Control : {len(both)}  {both}")

ASD sheet rows        : 1429
distinct variants     : 1425
rows by Status        : {'ASD': 1248, 'Control': 181}
listed as both ASD and Control : 4  ['1_202749094_G/A', '4_113354772_C/T', '7_70777116_G/A', '8_26647757_A/T']


## 3. Load Tim's ASD / BrainSpan file

One row per variant x transcript. Tim's file has no variant ID column, so we build the same
`chr_pos_REF/ALT` key from its coordinate columns.

In [3]:
tim = pd.read_csv(TIM_ASD, sep="\t").rename(columns={"MutPred2 score": "MutPred2_tim"})
tim["Variant"] = tim["#CHROM"].astype(str) + "_" + tim["POS"].astype(str) + "_" + tim["REF"] + "/" + tim["ALT"]

print(f"rows                 : {len(tim)}")
print(f"distinct variants    : {tim['Variant'].nunique()}")
print(f"distinct transcripts : {tim['transcript'].nunique()}")
print(f"rows with null score : {tim['MutPred2_tim'].isna().sum()}")

outside = set(tim["Variant"]) - ASD_VARIANTS
print(f"variants not on the ASD sheet : {len(outside)}")

rows                 : 1738
distinct variants    : 478
distinct transcripts : 547
rows with null score : 104
variants not on the ASD sheet : 0


### 3a. Are Tim's transcripts BrainSpan isoforms?

BrainSpan transcript IDs have no version suffix (`ENST...` vs `ENST....8`), so we strip the version
before comparing.

In [4]:
bs_tx = set(pd.read_csv(BrainSpan_expression_file_cortical, sep="\t", usecols=["transcript"])["transcript"])
tim_tx = set(tim["transcript"].str.split(".").str[0])
print(f"Tim's transcripts            : {len(tim_tx)}")
print(f"  found in BrainSpan (cortical): {len(tim_tx & bs_tx)}")

Tim's transcripts            : 547
  found in BrainSpan (cortical): 462


## 4. Question 1 — how many ASD variants does Tim's file score?

In [5]:
tim_scored   = set(tim.loc[tim["MutPred2_tim"].notna(), "Variant"]) & ASD_VARIANTS
tim_returned = set(tim["Variant"]) & ASD_VARIANTS
tim_all_null = tim_returned - tim_scored

n = len(ASD_VARIANTS)
print(f"ASD variants                               : {n}")
print(f"  present in Tim's file                    : {len(tim_returned):>5}  ({len(tim_returned)/n:.1%})")
print(f"  with >=1 non-null score  (= SCORABLE)    : {len(tim_scored):>5}  ({len(tim_scored)/n:.1%})")
print(f"  present but every transcript score null  : {len(tim_all_null):>5}")
print(f"  absent from Tim's file                   : {n - len(tim_returned):>5}")

ASD variants                               : 1425
  present in Tim's file                    :   478  (33.5%)
  with >=1 non-null score  (= SCORABLE)    :   448  (31.4%)
  present but every transcript score null  :    30
  absent from Tim's file                   :   947


Split by case / control status (a variant listed under both statuses counts in both rows).

In [6]:
def by_status(scored):
    return (asd.assign(scored=asd["Variant"].isin(scored))
               .groupby("Status")["scored"].agg(total="size", scored="sum")
               .assign(pct=lambda d: (d["scored"] / d["total"]).map("{:.1%}".format)))

by_status(tim_scored)

,total,scored,pct
Status,,,
ASD,1248,398,31.9%
Control,181,50,27.6%


## 5. Question 2 — how many ASD variants have a score in Tulika's file? (reference only)

Tulika's file keys variants in the `mutation` column, same `chr_pos_REF/ALT` format.

In [7]:
tulika = pd.read_csv(TULIKA_FILE, sep="\t")
tulika_scored = set(tulika.loc[tulika["mutpred"].notna(), "mutation"]) & ASD_VARIANTS

print(f"ASD variants with a score in Tulika's file : {len(tulika_scored):>5}  ({len(tulika_scored)/n:.1%})")
by_status(tulika_scored)

ASD variants with a score in Tulika's file :   498  (34.9%)


,total,scored,pct
Status,,,
ASD,1248,445,35.7%
Control,181,54,29.8%


## 6. Tim vs Tulika overlap

Tulika appears here only as provenance: how much of her ASD set we now hold from Tim's run, and which
of her variants are still a **gap** for us (candidates for a rescore request).

In [8]:
print(f"scored by both                 : {len(tim_scored & tulika_scored)}")
print(f"scored by Tim only             : {len(tim_scored - tulika_scored)}")
print(f"in Tulika only  (gap for us)   : {len(tulika_scored - tim_scored)}")
print(f"scored by neither              : {len(ASD_VARIANTS - tim_scored - tulika_scored)}")
print()
print(f"share of Tulika's ASD set reproduced by Tim : {len(tim_scored & tulika_scored)/len(tulika_scored):.1%}")

scored by both                 : 437
scored by Tim only             : 11
in Tulika only  (gap for us)   : 61
scored by neither              : 916

share of Tulika's ASD set reproduced by Tim : 87.8%


In [9]:
gap = (asd[asd["Variant"].isin(tulika_scored - tim_scored)]
         .assign(in_tim_file_but_null=lambda d: d["Variant"].isin(tim_all_null))
         [["Variant", "GeneSymbol", "Source", "Status", "in_tim_file_but_null"]]
         .sort_values(["GeneSymbol", "Variant"]))
print(f"{gap['Variant'].nunique()} variants; {gap.drop_duplicates('Variant')['in_tim_file_but_null'].sum()} of them are in Tim's file with only null scores")
gap

61 variants; 30 of them are in Tim's file with only null scores


,Variant,GeneSymbol,Source,Status,in_tim_file_but_null
1228,6_157181053_G/A,ARID1B,Fu,ASD,True
1237,6_157207431_A/C,ARID1B,Fu,ASD,True
608,1_155415921_A/C,ASH1L,Fu,ASD,False
994,3_10336173_T/C,ATP2B2,Fu,Control,False
1000,3_10358924_A/C,ATP2B2,"Antaki, Fu",ASD,False
...,...,...,...,...,...
1051,3_177047498_A/G,TBL1XR1,"Antaki, Fu",ASD,True
799,2_161417796_G/T,TBR1,Fu,ASD,True
1399,9_27203117_A/C,TEK,Fu,ASD,False
503,17_62600686_A/G,TLK2,Fu,ASD,True
